# BM25 top-10 000 → объединение → ближайшие 50 для benchmark

Для каждой из трёх моделей строим BM25 на **benchmark_items**:
`search_query → item_title_raw`, `search_query → item_description_raw`,
`search_infm_params_text → item_infm_params_text`. Каждая берёт top-10 000.
Объединяем выдачи по item_id и выбираем 50 ближайших к центру локации запроса.

Центры координат считаем только по train.parquet: медиана по item_location_id,
с fallback по связанным объявлениям search_location_id. BM25 индексирует именно
каталог, из которого нужно выдавать item_id. В выдаче каждый query_id встречается
ровно один раз, поле answer содержит до 50 item_id через пробел.

In [ ]:
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/retrievers/hybrid.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук из репозитория или задайте ROOT.")
TRAIN_PATH = ROOT / "data/train.parquet"
QUERY_PATH = ROOT / "data/benchmark_queries.parquet"
ITEM_PATH = ROOT / "data/benchmark_items.parquet"
N_RETRIEVAL = 10_000
K = 50
BATCH_SIZE = 16
LEMMATIZE = True
OUTPUT_DIR = ROOT / "artifacts/bm25_benchmark_top10k_geo"
CACHE_DIR = OUTPUT_DIR / "top50_cache"
OUTPUT_PATH = ROOT / "answer.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert N_RETRIEVAL >= K

In [ ]:
import gc
import hashlib
import json
import re
from time import perf_counter
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from src.split import split_queries
from src.geo import location_centers
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.hybrid import HybridRetriever

# Центры строим только по train-взаимодействиям.
train = pl.read_parquet(TRAIN_PATH, columns=["search_query", "search_location_id", "item_id", "item_location_id", "item_latitude", "item_longitude"])
train = train.with_columns(pl.Series("split", split_queries(train["search_query"], seed=42)))
centers = location_centers(pd.DataFrame(
    train.filter(pl.col("split") == "train")
    .select("search_location_id", "item_id", "item_location_id", "item_latitude", "item_longitude")
    .to_dict(as_series=False)
))
del train
gc.collect()

queries = pd.DataFrame(pl.read_parquet(QUERY_PATH).to_dict(as_series=False))
assert queries.query_id.notna().all() and queries.query_id.is_unique
assert queries.query_id.astype(str).str.len().eq(16).all()
queries = queries.join(centers, on="search_location_id")
items = pd.DataFrame(pl.read_parquet(ITEM_PATH, columns=["item_id", "item_location_id", "item_latitude", "item_longitude"])
                     .unique(subset="item_id", keep="last", maintain_order=True).to_dict(as_series=False))
assert items.item_id.is_unique and items.item_id.notna().all()
assert items.item_id.astype(str).str.fullmatch(r"[0-9a-f]{16}").all()
print("Queries:", len(queries), "unknown centers:", queries.center_lat.isna().sum())
print("Benchmark item corpus:", len(items), "items; valid coordinate pairs:",
      (pd.to_numeric(items.item_latitude, errors="coerce").between(-90, 90)
       & pd.to_numeric(items.item_longitude, errors="coerce").between(-180, 180)).sum())
display(queries.center_source.value_counts(dropna=False))

## Top-10 000 BM25 и геосортировка

Модели индексируют benchmark_items по одному, держим в памяти только текущий индекс.
Для каждого батча получаем три top-10 000, сливаем их и оставляем ближайшие 50.
Между этапами сохраняем лишь текущий top-50 на запрос, поэтому не создаём гигантскую
таблицу из 73 миллионов кандидатов. Кеш этапов позволяет продолжить прерванный запуск.

In [ ]:
from src.split import split_queries

sources = {
    "bm25_title": {"item_text_column": "item_title_raw", "query_text_column": "search_query"},
    "bm25_description": {"item_text_column": "item_description_raw", "query_text_column": "search_query"},
    "bm25_infm": {"item_text_column": "item_infm_params_text", "query_text_column": "search_infm_params_text"},
}
train_stat, query_stat, item_stat = TRAIN_PATH.stat(), QUERY_PATH.stat(), ITEM_PATH.stat()
query_hash = hashlib.sha256(pd.util.hash_pandas_object(queries, index=True).to_numpy().tobytes()).hexdigest()
signature = {
    "sources": sources, "lemmatize": LEMMATIZE, "k1": 1.5, "b": .75,
    "top_n": N_RETRIEVAL, "final_k": K, "batch_size": BATCH_SIZE, "split_seed": 42,
    "data": [[str(path.resolve()), stat.st_size, stat.st_mtime_ns] for path, stat in
             ((TRAIN_PATH, train_stat), (QUERY_PATH, query_stat), (ITEM_PATH, item_stat))],
    "queries": query_hash,
    "code": {name: hashlib.sha256((ROOT / "src/retrievers" / filename).read_bytes()).hexdigest()
             for name, filename in (("bm25", "bm25.py"), ("hybrid", "hybrid.py"))},
}
digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
run_cache = CACHE_DIR / digest
run_cache.mkdir(parents=True, exist_ok=True)
(run_cache / "manifest.json").write_text(json.dumps(signature, ensure_ascii=False, indent=2))

items_index = items.set_index("item_id")
hybrid = HybridRetriever(n_candidates=N_RETRIEVAL, distance_only=True).fit(items)
starts = list(range(0, len(queries), BATCH_SIZE))
for source_i, (name, params) in enumerate(sources.items()):
    source_dir = run_cache / name
    pending = [start for start in starts if not (source_dir / f"{start:07d}.parquet").is_file()]
    if not pending:
        print(name, "— cached top-50 ready")
        continue
    started = perf_counter()
    frame = (pl.scan_parquet(ITEM_PATH).select("item_id", params["item_text_column"])
             .unique(subset="item_id", keep="last", maintain_order=True).sort("item_id").collect())
    corpus = pd.DataFrame(frame.to_dict(as_series=False))
    model = BM25Retriever(**params, lemmatize=LEMMATIZE, batch_size=BATCH_SIZE).fit(corpus)
    del frame, corpus
    gc.collect()
    print(name, "index built in", round(perf_counter() - started, 1), "s")
    previous_dir = run_cache / list(sources)[source_i - 1] if source_i else None
    for batch_i, start in enumerate(pending, 1):
        batch = queries.iloc[start:start + BATCH_SIZE]
        retrieved = model.retrieve(batch, k=N_RETRIEVAL)
        features = hybrid.combine(batch, {name: retrieved})
        features["text_score"] = 0.0  # расстояние определяет порядок; ID разрешает ничьи
        if previous_dir is not None:
            previous = pd.DataFrame(pl.read_parquet(previous_dir / f"{start:07d}.parquet").to_dict(as_series=False))
            previous_features = previous[["query_index", "item_id", "distance_km"]].copy()
            previous_features["text_score"] = 0.0
            previous_features["same_location"] = False
            features = pd.concat([previous_features, features], ignore_index=True)
            features = features.drop_duplicates(["query_index", "item_id"], keep="first")
        if features.empty:
            checkpoint = pd.DataFrame({"query_index": pd.Series(dtype="int64"), "item_id": pd.Series(dtype="str"),
                                       "rank": pd.Series(dtype="int64"), "distance_km": pd.Series(dtype="float64")})
        else:
            ranked = hybrid.rerank(features, k=K)
            checkpoint = ranked.merge(
                features[["query_index", "item_id", "distance_km"]].drop_duplicates(["query_index", "item_id"]),
                on=["query_index", "item_id"], validate="one_to_one",
            )[["query_index", "item_id", "rank", "distance_km"]]
        path = source_dir / f"{start:07d}.parquet"
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp.parquet")
        pl.DataFrame(checkpoint.to_dict(orient="list"), schema={
            "query_index": pl.Int64, "item_id": pl.String, "rank": pl.Int64, "distance_km": pl.Float64,
        }).write_parquet(temporary)
        temporary.replace(path)
        if batch_i % 20 == 0 or batch_i == len(pending):
            print(name, f"{batch_i}/{len(pending)} batches; {perf_counter()-started:.1f}s")
    del model, retrieved, features, checkpoint
    gc.collect()

## Формируем и проверяем answer.csv

Файл записывается в UTF-8 с запятой-разделителем, ровно с колонками query_id и answer.
Каждый ответ сохраняет порядок по расстоянию, содержит не больше 50 уникальных item_id.

In [ ]:
final_dir = run_cache / list(sources)[-1]
ranked_parts = []
for start in starts:
    part = pd.DataFrame(pl.read_parquet(final_dir / f"{start:07d}.parquet").to_dict(as_series=False))
    if not part.empty:
        part["query_id"] = queries.iloc[start + part.query_index.to_numpy(dtype=int)].query_id.to_numpy()
        ranked_parts.append(part)
ranked = pd.concat(ranked_parts, ignore_index=True) if ranked_parts else pd.DataFrame(
    columns=["query_index", "item_id", "rank", "distance_km", "query_id"]
)
assert not ranked.duplicated(["query_id", "item_id"]).any()
assert ranked.item_id.astype(str).str.fullmatch(r"[0-9a-f]{16}").all()
assert ranked.item_id.isin(items.item_id).all()
assert ranked.groupby("query_id").size().le(K).all()
answers_by_query = ranked.sort_values(["query_index", "rank"]).groupby("query_id").item_id.agg(" ".join)
answer = queries[["query_id"]].copy()
answer["answer"] = answer.query_id.map(answers_by_query).fillna("")
assert answer.columns.tolist() == ["query_id", "answer"]
assert answer.query_id.is_unique and answer.query_id.tolist() == queries.query_id.tolist()
assert answer.query_id.str.len().eq(16).all()
assert answer.answer.map(lambda value: len(value.split()) <= K).all()
temporary = OUTPUT_PATH.with_suffix(".tmp.csv")
answer.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
temporary.replace(OUTPUT_PATH)

# Перечитываем ровно записанный файл и проверяем схему, покрытие и существование item_id.
check = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False, encoding="utf-8")
assert check.columns.tolist() == ["query_id", "answer"]
assert len(check) == len(queries) and check.query_id.is_unique
assert set(check.query_id) == set(queries.query_id)
valid_ids = set(items.item_id)
for values in check.answer:
    ids = values.split()
    assert len(ids) <= K and len(ids) == len(set(ids))
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) and item_id in valid_ids for item_id in ids)
print("Saved:", OUTPUT_PATH, "rows:", len(check), "predictions:", len(ranked))
display(check.head())